# CBP (Continual Backpropagation) Experiments

This notebook runs the Continual Backpropagation (CBP) algorithm on three benchmark datasets:
1. **Incremental CIFAR-100** - Class-incremental learning
2. **Permuted MNIST** - Online continual learning with permutations
3. **ImageNet** - Continual binary classification on ImageNet

CBP is a method designed to maintain plasticity in deep neural networks during continual learning by continuously reinitializing low-utility neurons.

## 1. Import Required Libraries and Setup

In [1]:
pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0+cu118, 2.2.1+cu118, 2.2.2+cu118, 2.3.0+cu118, 2.3.1+cu118, 2.4.0+cu118, 2.4.1+cu118, 2.5.0+cu118, 2.5.1+cu118, 2.6.0+cu118, 2.7.0+cu118, 2.7.1+cu118)
ERROR: No matching distribution found for torch==2.1.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install free-mujoco-py==2.1.6 gym==0.23.1 pycparser==2.21 zipp==3.16.2 sympy==1.9 setuptools==44.0.0 numpy==1.24.1 mlproj-manager==0.0.29 matplotlib==3.7.2 PyYAML==6.0.1 tqdm==4.66.1 scipy==1.11.2

ERROR: Ignored the following versions that require a different python version: 2.1.4 Requires-Python >=3.7.1,<3.11; 2.1.6 Requires-Python >=3.7.1,<3.11; 2.1.6.dev0 Requires-Python >=3.7.1,<3.11
ERROR: Could not find a version that satisfies the requirement free-mujoco-py==2.1.6 (from versions: none)
ERROR: No matching distribution found for free-mujoco-py==2.1.6
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install mlproj-manager==0.0.29

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 1.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import sys
import json
import time
import pickle
import copy
from copy import deepcopy
from functools import partialmethod

# Third-party libraries
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from tqdm import tqdm
import matplotlib.pyplot as plt

# Add project directories to path
sys.path.append("/kaggle/input/lop-src")
sys.path.append("/kaggle/input/lop-src/lop")
# Import custom modules
from lop.nets.torchvision_modified_resnet import build_resnet18, kaiming_init_resnet_module
from lop.utils.miscellaneous import nll_accuracy, compute_matrix_rank_summaries

# Import mlproj_manager for CIFAR-100 experiments
from mlproj_manager.problems import CifarDataSet
from mlproj_manager.util.data_preprocessing_and_transformations import (
    ToTensor, Normalize, RandomCrop, RandomHorizontalFlip, RandomRotator
)
# Setup device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seeds for reproducibility
random_seed = 42
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
np.random.seed(random_seed)

print("✓ Libraries imported successfully")

Using device: cuda:0
✓ Libraries imported successfully


## 2. Configuration and Utility Functions

In [5]:
# Create results directory
results_dir = "notebook_results"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(os.path.join(results_dir, "cifar100"), exist_ok=True)
os.makedirs(os.path.join(results_dir, "mnist"), exist_ok=True)
os.makedirs(os.path.join(results_dir, "imagenet"), exist_ok=True)

# Load configurations
def load_config(config_path):
    """Load configuration from JSON file"""
    with open(config_path, 'r') as f:
        return json.load(f)

# Save results function
def save_results(results, filename):
    """Save results to pickle file"""
    with open(filename, 'wb') as f:
        pickle.dump(results, f)
    print(f"Results saved to {filename}")

# Plotting function
def plot_learning_curve(accuracies, title, save_path=None):
    """Plot learning curves"""
    plt.figure(figsize=(10, 6))
    plt.plot(accuracies)
    plt.xlabel('Steps / Epochs')
    plt.ylabel('Accuracy')
    plt.title(title)
    plt.grid(True)
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

print("✓ Utility functions defined")

✓ Utility functions defined


## 3. Experiment 1: Incremental CIFAR-100 with CBP

This experiment demonstrates CBP on the incremental CIFAR-100 task, where classes are added incrementally over time.

In [6]:
# Load CBP configuration for CIFAR-100
cifar_config_path = '/kaggle/input/lop-src/lop/lop/incremental_cifar/cfg/continual_backpropagation.json'
cifar_config = load_config(cifar_config_path)

# Update config with notebook-specific paths
cifar_config["data_path"] = (lambda p: (os.makedirs(p, exist_ok=True), p)[1])("/kaggle/working/incremental_cifar/data")
cifar_config['results_dir'] = os.path.join(results_dir, 'cifar100')
cifar_config['num_workers'] = 2  # Adjust based on your system

print("CIFAR-100 CBP Configuration:")
print(f"  - Replacement rate: {cifar_config['replacement_rate']}")
print(f"  - Maturity threshold: {cifar_config['maturity_threshold']}")
print(f"  - Utility function: {cifar_config['utility_function']}")
print(f"  - Learning rate: {cifar_config['stepsize']}")
print(f"  - Weight decay: {cifar_config['weight_decay']}")
print(f"  - Momentum: {cifar_config['momentum']}")

CIFAR-100 CBP Configuration:
  - Replacement rate: 1e-05
  - Maturity threshold: 1000
  - Utility function: contribution
  - Learning rate: 0.1
  - Weight decay: 0.0005
  - Momentum: 0.9


In [7]:
# Prepare CIFAR-100 dataset using mlproj_manager (like in paper)
print("Loading CIFAR-100 dataset with mlproj_manager...")

os.makedirs(cifar_config['data_path'], exist_ok=True)

# Define transforms using mlproj_manager
mean = (0.5071, 0.4865, 0.4409)
std = (0.2673, 0.2564, 0.2762)

# Training transformations (with augmentation)
train_transformations = transforms.Compose([
    ToTensor(swap_color_axis=True),  # Reshape to (C x H x W)
    Normalize(mean=mean, std=std),
    RandomHorizontalFlip(p=0.5),
    RandomCrop(size=32, padding=4, padding_mode="reflect"),
    RandomRotator(degrees=(0, 15))
])

# Validation/Test transformations (no augmentation)
eval_transformations = transforms.Compose([
    ToTensor(swap_color_axis=True),
    Normalize(mean=mean, std=std)
])

# Load datasets with mlproj_manager's CifarDataSet
# This handles one-hot encoding and partitioning automatically
train_data_full = CifarDataSet(
    root_dir=cifar_config['data_path'],
    train=True,
    cifar_type=100,
    device=None,
    image_normalization="max",
    label_preprocessing="one-hot",
    use_torch=True
)

test_data = CifarDataSet(
    root_dir=cifar_config['data_path'],
    train=False,
    cifar_type=100,
    device=None,
    image_normalization="max",
    label_preprocessing="one-hot",
    use_torch=True
)

# Function to split train/validation like in the paper
def get_validation_and_train_indices(cifar_data, num_classes=100):
    """Split into 450 train + 50 validation per class"""
    num_val_per_class = 50
    num_train_per_class = 450
    val_size = 5000
    train_size = 45000
    
    val_indices = torch.zeros(val_size, dtype=torch.int32)
    train_indices = torch.zeros(train_size, dtype=torch.int32)
    
    current_val = 0
    current_train = 0
    
    for i in range(num_classes):
        # Find indices where class i has label 1 (one-hot encoded)
        class_indices = torch.argwhere(cifar_data.data["labels"][:, i] == 1).flatten()
        val_indices[current_val:current_val + num_val_per_class] = class_indices[:num_val_per_class]
        train_indices[current_train:current_train + num_train_per_class] = class_indices[num_val_per_class:]
        current_val += num_val_per_class
        current_train += num_train_per_class
    
    return train_indices, val_indices

# Get train/validation split
train_indices, val_indices = get_validation_and_train_indices(train_data_full)

# Helper to subsample dataset
def subsample_cifar(indices, cifar_data):
    """Subsample CIFAR dataset by indices"""
    cifar_data.data["data"] = cifar_data.data["data"][indices.numpy()]
    cifar_data.data["labels"] = cifar_data.data["labels"][indices.numpy()]
    cifar_data.integer_labels = torch.tensor(cifar_data.integer_labels)[indices.numpy()].tolist()
    cifar_data.current_data = cifar_data.partition_data()

# Create separate datasets for train and validation
import copy
train_data = copy.deepcopy(train_data_full)
val_data = copy.deepcopy(train_data_full)

subsample_cifar(train_indices, train_data)
subsample_cifar(val_indices, val_data)

# Set transformations
train_data.set_transformation(train_transformations)
val_data.set_transformation(eval_transformations)
test_data.set_transformation(eval_transformations)

print(f"✓ CIFAR-100 loaded with mlproj_manager:")
print(f"  - Train: {len(train_data.data['data'])} samples (450 per class)")
print(f"  - Validation: {len(val_data.data['data'])} samples (50 per class)")
print(f"  - Test: {len(test_data.data['data'])} samples")
print(f"  - Labels: one-hot encoded")
print(f"  - Using mlproj_manager's CifarDataSet class")

Loading CIFAR-100 dataset with mlproj_manager...


100%|██████████| 169M/169M [00:06<00:00, 27.4MB/s]


✓ CIFAR-100 loaded with mlproj_manager:
  - Train: 45000 samples (450 per class)
  - Validation: 5000 samples (50 per class)
  - Test: 10000 samples
  - Labels: one-hot encoded
  - Using mlproj_manager's CifarDataSet class


In [8]:
# Helper functions for metrics (matching paper's post_run_analysis.py)
# Import dormant neuron computation directly from the paper's code
from lop.incremental_cifar.post_run_analysis import compute_dormant_units_proportion

def compute_stable_rank(singular_values):
    """Compute stable rank using singular values (matching paper's post_run_analysis.py)
    
    Paper's definition: Number of singular values needed to capture 99% of total singular values
    This is an "effective rank" measure that indicates the dimensionality of the representation.
    """
    if len(singular_values) == 0:
        return 0
    
    # Sort singular values in descending order
    sorted_sv = np.flip(np.sort(singular_values))
    
    # Compute cumulative sum normalized by total
    cumsum_sv = np.cumsum(sorted_sv) / np.sum(singular_values)
    
    # Count how many SVs needed to reach 99%
    stable_rank = np.sum(cumsum_sv < 0.99) + 1
    
    return stable_rank

def compute_stable_rank_from_activations(activations):
    """Compute stable rank from layer activations using SVD"""
    try:
        # activations shape: [batch, features] or [batch, features, h, w]
        # Flatten to 2D if needed
        if activations.ndim > 2:
            # For conv layers: flatten spatial dimensions
            activations = activations.reshape(activations.shape[0], -1)
        
        # Check if we have valid data
        if activations.shape[0] == 0 or activations.shape[1] == 0:
            return 0
        
        # Compute SVD using scipy (more stable)
        # Note: compute_uv=False returns only singular values (not a tuple)
        from scipy.linalg import svd
        singular_values = svd(activations, compute_uv=False, lapack_driver="gesvd")
        
        return compute_stable_rank(singular_values)
    except Exception as e:
        print(f"Warning: SVD failed with error: {e}")
        return 0

@torch.no_grad()
def compute_avg_weight_magnitude(network):
    """Compute average magnitude of all weights (matching paper's post_run_analysis.py)"""
    num_weights = 0
    sum_weight_magnitude = 0.0
    
    for param in network.parameters():
        num_weights += param.numel()
        sum_weight_magnitude += torch.sum(torch.abs(param)).item()
    
    if num_weights == 0:
        return 0.0
    return sum_weight_magnitude / num_weights

## 3. Experiment: K-FAC Natural Gradient Descent
#
This experiment replaces standard SGD with **Kronecker-Factored Approximate
Curvature (K-FAC)** — an efficient Natural Gradient Descent method.
#
**Key idea**: Approximate the Fisher Information Matrix as a Kronecker product
$F \approx A \otimes G$, where:
- $A = \mathbb{E}[x x^T]$ (input covariance per layer)
- $G = \mathbb{E}[g g^T]$ (output gradient covariance per layer)
#
**Update rule**: $\Delta W = -\eta \cdot G^{-1} (\nabla_W \mathcal{L}) A^{-1}$
#
**Advantages over Empirical Fisher**:
- No per-sample gradient loop (hooks collect A and G automatically)
- Invert small matrices ($d_{in} \times d_{in}$ and $d_{out} \times d_{out}$) instead of one huge matrix
- Amortized inversion every $T_{inv}$ steps
#
**LoP mitigation**:
- **Dead Units**: Damping ensures update ≈ (1/√λ)·gradient when curvature → 0
- **Cloned Units**: Whitening via $A^{-1}$ and $G^{-1}$ breaks feature symmetry

### 3.1 K-FAC Optimizer Definition

In [9]:
# === K-FAC Natural Gradient Descent Optimizer ===

from torch.optim import Optimizer as _Optimizer

class KFACOptimizer(_Optimizer):
    """
    K-FAC (Kronecker-Factored Approximate Curvature) Optimizer.

    Approximates the Fisher Information Matrix for each layer as:
        F ≈ A ⊗ G
    where A = input covariance, G = output gradient covariance.

    Supports: nn.Linear, nn.Conv2d. Other layers fall back to SGD.
    BatchNorm parameters are always updated with plain SGD.

    For Conv2d layers, inputs are unfolded (im2col) to create a 2D
    representation suitable for Kronecker factorization.
    """

    def __init__(self, model, lr=0.01, damping=1e-3, weight_decay=0.0,
                 T_inv=100, alpha=0.95):
        """
        Args:
            model: nn.Module to optimize.
            lr: Learning rate.
            damping: Tikhonov damping λ. sqrt(λ) added to each factor.
            weight_decay: L2 regularization coefficient.
            T_inv: Frequency (in steps) to recompute matrix inverses.
            alpha: Exponential moving average coefficient for A and G.
        """
        self.model = model
        self.damping = damping
        self.weight_decay = weight_decay
        self.T_inv = T_inv
        self.alpha = alpha
        self.steps = 0

        # Storage for Kronecker factors and their inverses
        self._modules_tracked = {}  # name -> module
        self._stats = {}            # name -> {'A': Tensor, 'G': Tensor}
        self._inv = {}              # name -> {'A_inv': Tensor, 'G_inv': Tensor}
        self._hooks = []

        defaults = dict(lr=lr)
        super().__init__(model.parameters(), defaults)

        self._register_hooks()
        print(f"  K-FAC tracking {len(self._modules_tracked)} layers")

    def _register_hooks(self):
        """Register forward/backward hooks on Conv2d and Linear layers."""
        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Linear, nn.Conv2d)):
                self._modules_tracked[name] = module
                # Only initialize stats if not already present (preserve accumulated A/G)
                if name not in self._stats:
                    self._stats[name] = {'A': None, 'G': None}
                h1 = module.register_forward_hook(self._forward_hook(name, module))
                h2 = module.register_full_backward_hook(self._backward_hook(name, module))
                self._hooks.append(h1)
                self._hooks.append(h2)

    def _forward_hook(self, name, module):
        """Capture input activations → compute A = E[x x^T]."""
        def hook(mod, inp, out):
            if not mod.training:
                return
            with torch.no_grad():
                x = inp[0].detach()
                if isinstance(mod, nn.Conv2d):
                    # im2col: unfold input patches → [B*H'*W', C_in*k*k]
                    x = F.unfold(x, mod.kernel_size, dilation=mod.dilation,
                                 padding=mod.padding, stride=mod.stride)
                    # x shape: [B, C_in*k*k, L] → transpose → [B*L, C_in*k*k]
                    x = x.permute(0, 2, 1).reshape(-1, x.size(1))
                elif isinstance(mod, nn.Linear):
                    if x.dim() > 2:
                        x = x.reshape(-1, x.size(-1))

                # Append bias unit (column of 1s) if module has bias
                if mod.bias is not None:
                    ones = torch.ones(x.size(0), 1, device=x.device)
                    x = torch.cat([x, ones], dim=1)

                # A = x^T x / n
                n = x.size(0)
                cov_a = torch.matmul(x.t(), x) / n

                if self._stats[name]['A'] is None:
                    self._stats[name]['A'] = cov_a
                else:
                    self._stats[name]['A'].mul_(self.alpha).add_(cov_a, alpha=1 - self.alpha)
        return hook

    def _backward_hook(self, name, module):
        """Capture output gradients → compute G = E[g g^T]."""
        def hook(mod, grad_input, grad_output):
            if not mod.training:
                return
            with torch.no_grad():
                g = grad_output[0].detach()
                if isinstance(mod, nn.Conv2d):
                    # g shape: [B, C_out, H', W'] → [B*H'*W', C_out]
                    g = g.permute(0, 2, 3, 1).reshape(-1, g.size(1))
                elif isinstance(mod, nn.Linear):
                    if g.dim() > 2:
                        g = g.reshape(-1, g.size(-1))

                n = g.size(0)
                cov_g = torch.matmul(g.t(), g) / n

                if self._stats[name]['G'] is None:
                    self._stats[name]['G'] = cov_g
                else:
                    self._stats[name]['G'].mul_(self.alpha).add_(cov_g, alpha=1 - self.alpha)
        return hook

    @torch.no_grad()
    def _invert_factors(self):
        """Invert A and G with damping. Called every T_inv steps."""
        sqrt_damping = self.damping ** 0.5
        for name in self._stats:
            A = self._stats[name]['A']
            G = self._stats[name]['G']
            if A is None or G is None:
                continue
            try:
                A_d = A + sqrt_damping * torch.eye(A.size(0), device=A.device)
                G_d = G + sqrt_damping * torch.eye(G.size(0), device=G.device)
                self._inv[name] = {
                    'A_inv': torch.linalg.inv(A_d),
                    'G_inv': torch.linalg.inv(G_d)
                }
            except RuntimeError:
                pass  # Keep previous inverse if singular

    def reset_stats(self):
        """Reset running statistics (call at task boundaries)."""
        for name in self._stats:
            self._stats[name] = {'A': None, 'G': None}
        self._inv.clear()
        self.steps = 0

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        # Periodically recompute inverses
        if self.steps % self.T_inv == 0:
            self._invert_factors()

        # Apply K-FAC update to tracked layers
        for name, module in self._modules_tracked.items():
            if module.weight.grad is None:
                continue

            grad_w = module.weight.grad

            if name in self._inv:
                A_inv = self._inv[name]['A_inv']
                G_inv = self._inv[name]['G_inv']

                if isinstance(module, nn.Conv2d):
                    # Reshape weight grad: [C_out, C_in, k, k] → [C_out, C_in*k*k]
                    c_out = grad_w.size(0)
                    grad_2d = grad_w.reshape(c_out, -1)

                    if module.bias is not None and module.bias.grad is not None:
                        # Append bias grad as extra column
                        grad_2d = torch.cat([grad_2d, module.bias.grad.unsqueeze(1)], dim=1)

                    # K-FAC update: nat_grad = G_inv @ grad @ A_inv
                    nat_grad = torch.matmul(G_inv, torch.matmul(grad_2d, A_inv))

                    if module.bias is not None and module.bias.grad is not None:
                        # Split weight and bias natural gradients
                        nat_grad_w = nat_grad[:, :-1].reshape_as(module.weight)
                        nat_grad_b = nat_grad[:, -1]
                        if self.weight_decay > 0:
                            nat_grad_w.add_(module.weight, alpha=self.weight_decay)
                        module.weight.data.add_(nat_grad_w, alpha=-self.param_groups[0]['lr'])
                        module.bias.data.add_(nat_grad_b, alpha=-self.param_groups[0]['lr'])
                    else:
                        nat_grad = nat_grad.reshape_as(module.weight)
                        if self.weight_decay > 0:
                            nat_grad.add_(module.weight, alpha=self.weight_decay)
                        module.weight.data.add_(nat_grad, alpha=-self.param_groups[0]['lr'])

                elif isinstance(module, nn.Linear):
                    if module.bias is not None and module.bias.grad is not None:
                        # Append bias grad as extra column
                        grad_2d = torch.cat([grad_w, module.bias.grad.unsqueeze(1)], dim=1)
                    else:
                        grad_2d = grad_w

                    nat_grad = torch.matmul(G_inv, torch.matmul(grad_2d, A_inv))

                    if module.bias is not None and module.bias.grad is not None:
                        nat_grad_w = nat_grad[:, :-1]
                        nat_grad_b = nat_grad[:, -1]
                        if self.weight_decay > 0:
                            nat_grad_w.add_(module.weight, alpha=self.weight_decay)
                        module.weight.data.add_(nat_grad_w, alpha=-self.param_groups[0]['lr'])
                        module.bias.data.add_(nat_grad_b, alpha=-self.param_groups[0]['lr'])
                    else:
                        if self.weight_decay > 0:
                            nat_grad.add_(module.weight, alpha=self.weight_decay)
                        module.weight.data.add_(nat_grad, alpha=-self.param_groups[0]['lr'])
            else:
                # No inverse available yet → plain SGD fallback
                if self.weight_decay > 0:
                    grad_w = grad_w + self.weight_decay * module.weight
                module.weight.data.add_(grad_w, alpha=-self.param_groups[0]['lr'])
                if module.bias is not None and module.bias.grad is not None:
                    module.bias.data.add_(module.bias.grad, alpha=-self.param_groups[0]['lr'])

        # SGD for non-tracked parameters (BatchNorm etc.)
        tracked_params = set()
        for mod in self._modules_tracked.values():
            tracked_params.add(id(mod.weight))
            if mod.bias is not None:
                tracked_params.add(id(mod.bias))

        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None or id(p) in tracked_params:
                    continue
                if self.weight_decay > 0:
                    p.data.add_(p, alpha=-self.weight_decay * group['lr'])
                p.data.add_(p.grad, alpha=-group['lr'])

        self.steps += 1
        return loss

print("✓ KFACOptimizer defined (supports Conv2d + Linear + BN fallback)")

✓ KFACOptimizer defined (supports Conv2d + Linear + BN fallback)


### 3.2 K-FAC Experiment Configuration & Training

In [10]:
# Configuration for K-FAC experiment
num_classes = 100  # CIFAR-100
kfac_config = {
    'num_epochs': 1200,
    'lr': 0.01,                 # LR — smaller due to Fisher preconditioning
    'damping': 1e-3,            # λ: sqrt(λ) added to A and G before inversion
    'weight_decay': 5e-4,
    'T_inv': 100,               # Recompute A^{-1}, G^{-1} every T_inv steps
    'alpha': 0.95,              # EMA coefficient for running A and G stats
    'batch_size': 90,
    'method': 'kfac_natural_gradient',
    'class_increase_frequency': 200,
    'checkpoint_frequency': 500,
    'random_seed': 42
}

print("\nK-FAC (Kronecker-Factored) Configuration:")
print(f"  - Method: {kfac_config['method']}")
print(f"  - LR: {kfac_config['lr']}")
print(f"  - Damping: {kfac_config['damping']}")
print(f"  - T_inv: {kfac_config['T_inv']} (inversion frequency)")
print(f"  - Alpha: {kfac_config['alpha']} (EMA for stats)")
print(f"  - Num epochs: {kfac_config['num_epochs']}")


K-FAC (Kronecker-Factored) Configuration:
  - Method: kfac_natural_gradient
  - LR: 0.01
  - Damping: 0.001
  - T_inv: 100 (inversion frequency)
  - Alpha: 0.95 (EMA for stats)
  - Num epochs: 1200


In [11]:
# Initialize model for K-FAC experiment
print("\nInitializing ResNet-18 for K-FAC experiment...")
net_ngd = build_resnet18(num_classes=num_classes, norm_layer=nn.BatchNorm2d)
net_ngd.apply(kaiming_init_resnet_module)
net_ngd.to(device)

# K-FAC optimizer for ALL layers (Conv2d + Linear via hooks, BN via SGD fallback)
optim_ngd = KFACOptimizer(
    net_ngd,
    lr=kfac_config['lr'],
    damping=kfac_config['damping'],
    weight_decay=kfac_config['weight_decay'],
    T_inv=kfac_config['T_inv'],
    alpha=kfac_config['alpha']
)

loss_fn_ngd = nn.CrossEntropyLoss()

total_params = sum(p.numel() for p in net_ngd.parameters())
fc_params_count = sum(p.numel() for p in net_ngd.fc.parameters())
conv_params_count = total_params - fc_params_count
print(f"✓ Total params: {total_params:,}")
print(f"  - Conv/BN (K-FAC for Conv2d, SGD for BN): {conv_params_count:,}")
print(f"  - FC (K-FAC with full Kronecker):          {fc_params_count:,}")
print(f"✓ Model initialized for K-FAC")


Initializing ResNet-18 for K-FAC experiment...
  K-FAC tracking 21 layers
✓ Total params: 11,224,932
  - Conv/BN (K-FAC for Conv2d, SGD for BN): 11,173,632
  - FC (K-FAC with full Kronecker):          51,300
✓ Model initialized for K-FAC


In [12]:
# Setup data loaders for NGD
current_num_classes_ngd = 5
all_classes_ngd = np.random.RandomState(kfac_config['random_seed']).permutation(num_classes)

train_data_ngd = copy.deepcopy(train_data)
val_data_ngd = copy.deepcopy(val_data)
test_data_ngd = copy.deepcopy(test_data)

train_data_ngd.select_new_partition(all_classes_ngd[:current_num_classes_ngd])
val_data_ngd.select_new_partition(all_classes_ngd[:current_num_classes_ngd])
test_data_ngd.select_new_partition(all_classes_ngd[:current_num_classes_ngd])

train_loader_ngd = DataLoader(train_data_ngd, batch_size=kfac_config['batch_size'], shuffle=True, num_workers=0)
val_loader_ngd = DataLoader(val_data_ngd, batch_size=50, shuffle=False, num_workers=0)
test_loader_ngd = DataLoader(test_data_ngd, batch_size=100, shuffle=False, num_workers=0)

# Separate data loaders for dormant unit measurement (matching paper's post_run_analysis.py)
# Paper measures TWO metrics:
#   dormant_after = dormant on NEXT task data (unseen) → measures plasticity
#   dormant_before = dormant on ALL PREVIOUS tasks data (already trained)

# Dormant "after" loader — next task classes (unseen)
dormant_next_data = copy.deepcopy(train_data_full)
subsample_cifar(train_indices, dormant_next_data)
dormant_next_data.set_transformation(eval_transformations)  # No augmentation!
next_start_d = current_num_classes_ngd  # 5
next_end_d = min(current_num_classes_ngd + 5, num_classes)  # 10
if next_start_d < num_classes:
    dormant_next_data.select_new_partition(all_classes_ngd[next_start_d:next_end_d])
    dormant_next_loader = DataLoader(dormant_next_data, batch_size=1000, shuffle=True, num_workers=0)
else:
    dormant_next_loader = None

# Dormant "before" loader — all previous tasks' classes (none at start for task 0)
dormant_prev_data = copy.deepcopy(train_data_full)
subsample_cifar(train_indices, dormant_prev_data)
dormant_prev_data.set_transformation(eval_transformations)  # No augmentation!
# At start (task 0), there are no previous tasks
dormant_prev_loader = None

print(f"✓ Data loaders initialized with {current_num_classes_ngd} classes")
print(f"✓ Dormant measurement: next_task + previous_tasks loaders (matching paper's post_run_analysis.py)")

✓ Data loaders initialized with 5 classes
✓ Dormant measurement: next_task + previous_tasks loaders (matching paper's post_run_analysis.py)


In [13]:
# K-FAC Training Loop
num_epochs_ngd = kfac_config['num_epochs']
class_increase_frequency_ngd = kfac_config['class_increase_frequency']
checkpoint_frequency_ngd = kfac_config['checkpoint_frequency']

checkpoint_dir_ngd = os.path.join(results_dir, "cifar100_ngd", "checkpoints")
os.makedirs(checkpoint_dir_ngd, exist_ok=True)

# Metric storage
train_losses_ngd = []
train_accuracies_ngd = []
val_accuracies_per_epoch_ngd = []
test_accuracies_per_epoch_ngd = []
dormant_after_per_epoch_ngd = []   # dormant on next task data (plasticity measure)
dormant_before_per_epoch_ngd = []  # dormant on all previous tasks data
stable_ranks_per_task_ngd = []
avg_weight_magnitude_per_epoch_ngd = []
epoch_runtimes_ngd = []
fisher_stats_ngd = []  # Track Fisher matrix condition numbers

print(f"\nStarting K-FAC training for {num_epochs_ngd} epochs...")
print(f"Incremental: start with {current_num_classes_ngd} classes, add 5 every {class_increase_frequency_ngd} epochs")
print(f"K-FAC applied to ALL Conv2d+Linear layers, BN uses SGD")

net_ngd.train()
for epoch in range(num_epochs_ngd):
    epoch_start_time = time.time()

    # Measure stable rank at task boundaries
    if epoch % class_increase_frequency_ngd == 0:
        next_task_start = current_num_classes_ngd
        next_task_end = min(current_num_classes_ngd + 5, num_classes)
        next_task_classes = all_classes_ngd[next_task_start:next_task_end]

        if len(next_task_classes) > 0:
            temp_data_ngd = copy.deepcopy(train_data_ngd)
            temp_data_ngd.select_new_partition(next_task_classes)
            temp_loader_ngd = DataLoader(temp_data_ngd, batch_size=kfac_config['batch_size'], shuffle=False, num_workers=0)

            net_ngd.eval()
            all_activations_ngd = []
            samples_collected_ngd = 0
            target_samples_ngd = 1000

            with torch.no_grad():
                for sample in temp_loader_ngd:
                    if samples_collected_ngd >= target_samples_ngd:
                        break
                    images = sample["image"].to(device)
                    features_per_layer = []
                    net_ngd.forward(images, features_per_layer)
                    all_activations_ngd.append(features_per_layer[-1].cpu())
                    samples_collected_ngd += images.shape[0]

            if len(all_activations_ngd) > 0:
                last_layer_act_ngd = torch.cat(all_activations_ngd, dim=0)[:target_samples_ngd].numpy()
                stable_rank_val_ngd = compute_stable_rank_from_activations(last_layer_act_ngd)
                stable_ranks_per_task_ngd.append(stable_rank_val_ngd)
                print(f"  → Stable Rank (before training task {len(stable_ranks_per_task_ngd)}): {stable_rank_val_ngd:.2f} ({stable_rank_val_ngd/512*100:.1f}%)")

            net_ngd.train()

    # Add new classes at task boundaries
    if epoch > 0 and (epoch % class_increase_frequency_ngd) == 0 and current_num_classes_ngd < num_classes:
        current_num_classes_ngd = min(current_num_classes_ngd + 5, num_classes)
        train_data_ngd.select_new_partition(all_classes_ngd[:current_num_classes_ngd])
        val_data_ngd.select_new_partition(all_classes_ngd[:current_num_classes_ngd])
        test_data_ngd.select_new_partition(all_classes_ngd[:current_num_classes_ngd])

        train_loader_ngd = DataLoader(train_data_ngd, batch_size=kfac_config['batch_size'], shuffle=True, num_workers=0)
        val_loader_ngd = DataLoader(val_data_ngd, batch_size=50, shuffle=False, num_workers=0)
        test_loader_ngd = DataLoader(test_data_ngd, batch_size=100, shuffle=False, num_workers=0)

        print(f"\n{'='*60}")
        print(f"K-FAC: New task at epoch {epoch} → {current_num_classes_ngd} classes")
        print(f"{'='*60}")

        # Reset K-FAC statistics (stale after distribution shift)
        optim_ngd.reset_stats()

        # Update dormant loaders (matching paper's post_run_analysis.py)
        # dormant_after = next task (unseen classes)
        next_start_d = current_num_classes_ngd
        next_end_d = min(current_num_classes_ngd + 5, num_classes)
        if next_start_d < num_classes:
            dormant_next_data.select_new_partition(all_classes_ngd[next_start_d:next_end_d])
            dormant_next_loader = DataLoader(dormant_next_data, batch_size=1000, shuffle=True, num_workers=0)
        else:
            dormant_next_loader = None  # last task, no next task
        # dormant_before = all previous tasks (exclude current task)
        prev_end_d = current_num_classes_ngd - 5
        if prev_end_d > 0:
            dormant_prev_data.select_new_partition(all_classes_ngd[:prev_end_d])
            dormant_prev_loader = DataLoader(dormant_prev_data, batch_size=1000, shuffle=True, num_workers=0)
        else:
            dormant_prev_loader = None

    print(f"\nEpoch {epoch+1}/{num_epochs_ngd} | Classes: {current_num_classes_ngd}")

    # Training
    running_loss_ngd = 0
    running_acc_ngd = 0
    num_batches_ngd = 0

    for sample in tqdm(train_loader_ngd, desc=f"K-FAC epoch {epoch+1}"):
        images = sample["image"].to(device)
        labels = sample["label"].to(device)

        # --- Forward + backward (hooks auto-collect A and G) ---
        net_ngd.train()
        predictions = net_ngd(images)
        predictions_masked = predictions[:, all_classes_ngd[:current_num_classes_ngd]]

        # Handle one-hot labels
        loss = loss_fn_ngd(predictions_masked, labels.argmax(dim=1) if labels.dim() > 1 and labels.shape[1] > 1 else labels)

        optim_ngd.zero_grad()
        loss.backward()

        # --- K-FAC step: preconditioned update for all layers ---
        # Hooks already captured A (input cov) and G (grad cov)
        # Inverses recomputed every T_inv steps automatically
        optim_ngd.step()

        # Metrics
        with torch.no_grad():
            acc = (predictions_masked.argmax(dim=1) == (labels.argmax(dim=1) if labels.dim() > 1 and labels.shape[1] > 1 else labels)).float().mean()
        running_loss_ngd += loss.item()
        running_acc_ngd += acc.item()
        num_batches_ngd += 1

    # Store metrics
    train_losses_ngd.append(running_loss_ngd / num_batches_ngd)
    train_accuracies_ngd.append(running_acc_ngd / num_batches_ngd)

    # Validation
    net_ngd.eval()
    val_acc_ngd = 0
    val_batches_ngd = 0
    with torch.no_grad():
        for sample in val_loader_ngd:
            images = sample["image"].to(device)
            labels = sample["label"].to(device)
            predictions = net_ngd(images)
            predictions_masked = predictions[:, all_classes_ngd[:current_num_classes_ngd]]
            acc = (predictions_masked.argmax(dim=1) == (labels.argmax(dim=1) if labels.dim() > 1 and labels.shape[1] > 1 else labels)).float().mean()
            val_acc_ngd += acc.item()
            val_batches_ngd += 1
    val_accuracies_per_epoch_ngd.append(val_acc_ngd / val_batches_ngd if val_batches_ngd > 0 else 0)

    # Test
    test_acc_ngd = 0
    test_batches_ngd = 0
    with torch.no_grad():
        for sample in test_loader_ngd:
            images = sample["image"].to(device)
            labels = sample["label"].to(device)
            predictions = net_ngd(images)
            predictions_masked = predictions[:, all_classes_ngd[:current_num_classes_ngd]]
            acc = (predictions_masked.argmax(dim=1) == (labels.argmax(dim=1) if labels.dim() > 1 and labels.shape[1] > 1 else labels)).float().mean()
            test_acc_ngd += acc.item()
            test_batches_ngd += 1
    test_accuracies_per_epoch_ngd.append(test_acc_ngd / test_batches_ngd if test_batches_ngd > 0 else 0)

    # Plasticity metrics — matching paper's post_run_analysis.py methodology:
    # Paper does this as POST-RUN analysis on saved checkpoints (not during training).
    # We do it inline for convenience, but must avoid corrupting K-FAC statistics.
    
    # Temporarily remove K-FAC hooks so they don't fire during dormant measurement
    for h in optim_ngd._hooks:
        h.remove()
    optim_ngd._hooks.clear()
    
    # Measure dormant units (matching paper's post_run_analysis.py methodology)
    # Paper measures on next task (unseen) and all previous tasks (already trained)
    net_ngd.train()
    
    # dormant_after: next task data (plasticity)
    if dormant_next_loader is not None:
        dormant_after_ngd, _ = compute_dormant_units_proportion(net_ngd, dormant_next_loader, dormant_unit_threshold=0.01)
    else:
        dormant_after_ngd = float('nan')
    dormant_after_per_epoch_ngd.append(dormant_after_ngd)
    
    # dormant_before: all previous tasks data
    if dormant_prev_loader is not None:
        dormant_before_ngd, _ = compute_dormant_units_proportion(net_ngd, dormant_prev_loader, dormant_unit_threshold=0.01)
    else:
        dormant_before_ngd = float('nan')
    dormant_before_per_epoch_ngd.append(dormant_before_ngd)
    
    # Debug: show per-layer activation sparsity every 50 epochs
    if epoch % 50 == 0:
        with torch.no_grad():
            dbg_loader = dormant_next_loader if dormant_next_loader is not None else dormant_prev_loader
            for sample in (dbg_loader or []):
                img_dbg = sample["image"].to(device)
                feats_dbg = []
                net_ngd.forward(img_dbg, feats_dbg)
                print(f"  [DEBUG] Dormant analysis on {img_dbg.shape[0]} samples:")
                for li, f in enumerate(feats_dbg):
                    zero_frac = (f == 0).float().mean().item()
                    if f.dim() == 4:
                        per_ch = (f != 0).float().mean(dim=(0,2,3))
                        dormant_ch = (per_ch < 0.01).sum().item()
                        print(f"    Layer {li}: shape={list(f.shape)}, zero_frac={zero_frac:.3f}, dormant_channels={dormant_ch}/{f.shape[1]}")
                    else:
                        per_unit = (f != 0).float().mean(dim=0)
                        dormant_units = (per_unit < 0.01).sum().item()
                        print(f"    Layer {li} (FC): shape={list(f.shape)}, zero_frac={zero_frac:.3f}, dormant_units={dormant_units}/{f.shape[1]}")
                break
    
    # Re-register K-FAC hooks
    optim_ngd._register_hooks()
    
    avg_weight_mag_ngd = compute_avg_weight_magnitude(net_ngd)
    avg_weight_magnitude_per_epoch_ngd.append(avg_weight_mag_ngd)

    net_ngd.train()

    epoch_runtime_ngd = time.time() - epoch_start_time
    epoch_runtimes_ngd.append(epoch_runtime_ngd)

    print(f"  Train: {train_accuracies_ngd[-1]:.4f} | Val: {val_accuracies_per_epoch_ngd[-1]:.4f} | Test: {test_accuracies_per_epoch_ngd[-1]:.4f}")
    print(f"  Loss: {train_losses_ngd[-1]:.4f} | Dormant(next): {dormant_after_ngd:.4f} | Dormant(prev): {dormant_before_ngd:.4f} | Avg Weight: {avg_weight_mag_ngd:.4f}")
    print(f"  Epoch time: {epoch_runtime_ngd:.1f}s")

    # Checkpoint
    if (epoch + 1) % checkpoint_frequency_ngd == 0 or (epoch + 1) == num_epochs_ngd:
        checkpoint_path_ngd = os.path.join(checkpoint_dir_ngd, f'checkpoint_epoch_{epoch+1}.pt')
        checkpoint_data_ngd = {
            'epoch': epoch + 1,
            'model_state_dict': net_ngd.state_dict(),
            'current_num_classes': current_num_classes_ngd,
            'train_losses': train_losses_ngd,
            'train_accuracies': train_accuracies_ngd,
            'val_accuracies': val_accuracies_per_epoch_ngd,
            'test_accuracies': test_accuracies_per_epoch_ngd,
            'dormant_after_per_epoch': dormant_after_per_epoch_ngd,
            'dormant_before_per_epoch': dormant_before_per_epoch_ngd,
            'stable_ranks': stable_ranks_per_task_ngd,
            'config': kfac_config
        }
        torch.save(checkpoint_data_ngd, checkpoint_path_ngd)
        print(f"  → Checkpoint saved: {checkpoint_path_ngd}")

print(f"\n✓ K-FAC training completed!")


Starting K-FAC training for 1200 epochs...
Incremental: start with 5 classes, add 5 every 200 epochs
K-FAC applied to ALL Conv2d+Linear layers, BN uses SGD
  → Stable Rank (before training task 1): 458.00 (89.5%)

Epoch 1/1200 | Classes: 5


K-FAC epoch 1: 100%|██████████| 25/25 [00:03<00:00,  6.99it/s]


  [DEBUG] Dormant analysis on 1000 samples:
    Layer 0: shape=[1000, 64, 32, 32], zero_frac=0.515, dormant_channels=0/64
    Layer 1: shape=[1000, 64, 32, 32], zero_frac=0.539, dormant_channels=0/64
    Layer 2: shape=[1000, 64, 32, 32], zero_frac=0.394, dormant_channels=0/64
    Layer 3: shape=[1000, 64, 32, 32], zero_frac=0.496, dormant_channels=0/64
    Layer 4: shape=[1000, 64, 32, 32], zero_frac=0.306, dormant_channels=0/64
    Layer 5: shape=[1000, 128, 16, 16], zero_frac=0.526, dormant_channels=0/128
    Layer 6: shape=[1000, 128, 16, 16], zero_frac=0.558, dormant_channels=0/128
    Layer 7: shape=[1000, 128, 16, 16], zero_frac=0.505, dormant_channels=0/128
    Layer 8: shape=[1000, 128, 16, 16], zero_frac=0.347, dormant_channels=0/128
    Layer 9: shape=[1000, 256, 8, 8], zero_frac=0.492, dormant_channels=0/256
    Layer 10: shape=[1000, 256, 8, 8], zero_frac=0.553, dormant_channels=0/256
    Layer 11: shape=[1000, 256, 8, 8], zero_frac=0.612, dormant_channels=0/256
    Layer 

K-FAC epoch 2: 100%|██████████| 25/25 [00:01<00:00, 12.68it/s]


  Train: 0.3724 | Val: 0.4920 | Test: 0.4960
  Loss: 8.3655 | Dormant(next): 0.0000 | Dormant(prev): nan | Avg Weight: 0.1181
  Epoch time: 2.1s

Epoch 3/1200 | Classes: 5


...

  Train: 0.9956 | Val: 0.6480 | Test: 0.6540
  Loss: 0.0146 | Dormant(next): 0.0000 | Dormant(prev): 0.0000 | Avg Weight: 0.0881
  Epoch time: 11.5s

Epoch 1199/1200 | Classes: 30


K-FAC epoch 1199: 100%|██████████| 150/150 [00:11<00:00, 13.37it/s]


  Train: 0.9960 | Val: 0.6487 | Test: 0.6500
  Loss: 0.0170 | Dormant(next): 0.0000 | Dormant(prev): 0.0000 | Avg Weight: 0.0880
  Epoch time: 11.6s

Epoch 1200/1200 | Classes: 30


K-FAC epoch 1200: 100%|██████████| 150/150 [00:11<00:00, 13.55it/s]


  Train: 0.9960 | Val: 0.6500 | Test: 0.6560
  Loss: 0.0135 | Dormant(next): 0.0000 | Dormant(prev): 0.0000 | Avg Weight: 0.0880
  Epoch time: 11.5s
  → Checkpoint saved: notebook_results/cifar100_ngd/checkpoints/checkpoint_epoch_1200.pt

✓ K-FAC training completed!


In [14]:
# Save K-FAC results
ngd_results = {
    'train_losses': train_losses_ngd,
    'train_accuracies': train_accuracies_ngd,
    'val_accuracy_per_epoch': val_accuracies_per_epoch_ngd,
    'test_accuracy_per_epoch': test_accuracies_per_epoch_ngd,
    'epoch_runtime': epoch_runtimes_ngd,
    'dormant_after_per_epoch': dormant_after_per_epoch_ngd,
    'dormant_before_per_epoch': dormant_before_per_epoch_ngd,
    'stable_rank_per_task': stable_ranks_per_task_ngd,
    'avg_weight_magnitude_per_epoch': avg_weight_magnitude_per_epoch_ngd,
    'config': kfac_config,
    'class_order': all_classes_ngd.tolist()
}

os.makedirs(os.path.join(results_dir, "cifar100_kfac"), exist_ok=True)
save_results(ngd_results, os.path.join(results_dir, 'cifar100_kfac', 'kfac_results.pkl'))

print(f"\n✓ Saved K-FAC metrics:")
print(f"  - Epochs: {len(test_accuracies_per_epoch_ngd)}")
print(f"  - Dormant analysis: {len(dormant_after_per_epoch_ngd)} after (next task) + {len(dormant_before_per_epoch_ngd)} before (prev tasks)")
print(f"  - Stable rank analysis: {len(stable_ranks_per_task_ngd)} measurements")

Results saved to notebook_results/cifar100_kfac/kfac_results.pkl

✓ Saved K-FAC metrics:
  - Epochs: 1200
  - Dormant analysis: 1200 after (next task) + 1200 before (prev tasks)
  - Stable rank analysis: 6 measurements
